# Notebook 00 - Inspecting and Cleaning the Raw Data

This notebook reads three months of NYC Yellow Taxi trips (January to March 2024) from HDFS, checks their quality, applies the cleaning rules and writes one clean base back to HDFS. Every format and precision level in the next notebooks is built from this base, so every experiment uses exactly the same trips.

**Input** `hdfs:///ca1/raw` (three monthly Parquet files and the zone lookup table). **Output** `hdfs:///ca1/clean/yellow_2024q1`.

### Step 1 - Import Libraries

In [1]:
# PySpark
import pyspark                                    
from pyspark.sql import SparkSession              
from pyspark.sql import functions as F            
print("PySpark: %s" % pyspark.__version__)        

PySpark: 4.2.0


### Step 2 - Data Source

The trips come from the NYC Taxi and Limousine Commission (TLC) as monthly Parquet files, together with the zone lookup table `taxi_zone_lookup.csv`. The files were downloaded on 29 September 2026.

There is no named licence. The NYC Open Data Law allows free use as long as the source is named and the changes are described, and the changes made here are the cleaning rules in Step 6. The data has no driver, vehicle or passenger identifiers, so there is no personal data.

### Step 3 - Spark Session and Raw Files

Spark runs in local mode with the 4 cores of the VM and 4 GB for the driver, and reads from HDFS in pseudo-distributed mode. The session time zone is UTC. The timestamps have no time zone, and with the VM's zone (Europe/Dublin) the clock change on 31 March could shift some durations by one hour.

In [2]:
spark = (SparkSession.builder
         .master("local[4]")                               # the 4 cores of the VM
         .appName("ca1-00-inspect")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.session.timeZone", "UTC")      # no clock change on 31 March (on summertime)
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")                   

RAW = "hdfs://localhost:9000/ca1/raw"
CLEAN = "hdfs://localhost:9000/ca1/clean/yellow_2024q1"

df = spark.read.parquet(f"{RAW}/yellow_tripdata_2024-*.parquet")   
df.printSchema()
print("Trips:", df.count())

dur = F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/08 08:08:56 WARN Utils: Your hostname, BDSP, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/08 08:08:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/08 08:08:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



Trips: 9554778


The three monthly files share the same schema, so one path with a wildcard reads all of them. Together they hold 9,554,778 trips and 19 columns. Pickup and dropoff are `timestamp_ntz`, distance and fares are `double` and the zone IDs are `integer`.

### Step 4 - Quality Checks

Before cleaning anything, the cell checks the date range, the share of trips with zero or negative distance or duration, the share longer than 4 hours, and the median and 99th percentile of the distance.

In [3]:
df.select(F.min("tpep_pickup_datetime").alias("first_pickup"),
          F.max("tpep_pickup_datetime").alias("last_pickup"),
          F.avg((F.col("trip_distance") <= 0).cast("int")).alias("share_distance_zero_or_negative"),
          F.avg((dur <= 0).cast("int")).alias("share_duration_zero_or_negative"),
          F.avg((dur > 4 * 3600).cast("int")).alias("share_longer_than_4h"),
          F.expr("percentile_approx(trip_distance, array(0.5, 0.99))").alias("distance_median_p99")
         ).show(vertical=True, truncate=False)

-RECORD 0------------------------------------------------
 first_pickup                    | 2002-12-31 22:17:10   
 last_pickup                     | 2024-04-01 00:34:55   
 share_distance_zero_or_negative | 0.022581791015971277  
 share_duration_zero_or_negative | 2.9315176134913864E-4 
 share_longer_than_4h            | 5.977114277275726E-4  
 distance_median_p99             | [1.7, 19.87]          



2.26% of the trips have zero or negative distance, 0.03% have zero or negative duration and 0.06% last more than 4 hours. The median distance is 1.7 miles and the 99th percentile is 19.87 miles, so most trips are short.

### Step 5 - Missing Values and Unknown Zones

This cell measures the share of nulls in the columns that could be features, the largest distance, and how many trips start or end in the two unknown zones (IDs 264 and 265 in the lookup table).

In [4]:
cols = ["passenger_count", "RatecodeID", "PULocationID", "DOLocationID", "trip_distance"]
df.select([F.avg(F.col(c).isNull().cast("int")).alias(c) for c in cols]).show(vertical=True)

# Largest distance / share of trips in the unknown zones
df.select(F.max("trip_distance").alias("max_distance"),
          F.avg(F.col("PULocationID").isin(264, 265).cast("int")).alias("share_unknown_pickup"),
          F.avg(F.col("DOLocationID").isin(264, 265).cast("int")).alias("share_unknown_dropoff")
         ).show(vertical=True)

zones = spark.read.option("header", True).csv(f"{RAW}/taxi_zone_lookup.csv")
zones.filter(F.col("LocationID").cast("int") >= 264).show(truncate=False)

-RECORD 0------------------------------
 passenger_count | 0.07870010166641235 
 RatecodeID      | 0.07870010166641235 
 PULocationID    | 0.0                 
 DOLocationID    | 0.0                 
 trip_distance   | 0.0                 



-RECORD 0-------------------------------------
 max_distance          | 312722.3             
 share_unknown_pickup  | 0.003847289806210... 
 share_unknown_dropoff | 0.008883408908087661 



+----------+-------+--------------+------------+
|LocationID|Borough|Zone          |service_zone|
+----------+-------+--------------+------------+
|264       |Unknown|N/A           |N/A         |
|265       |N/A    |Outside of NYC|N/A         |
+----------+-------+--------------+------------+



`passenger_count` and `RatecodeID` are missing in 7.87% of the trips. They are not used by the network, so nothing needs to be imputed. The largest distance is 312,722.3 miles, a clear recording error. 0.38% of the pickups and 0.89% of the dropoffs are in the two unknown zones.

The network uses the distance, the hour and day of week of the pickup, and the pickup and dropoff zones.

### Step 6 - Cleaning Rules

The rules run one after the other, and each line shows how many trips it removes.

- Pickup between 1 January and 31 March 2024.
- Distance above 0 and up to 100 miles. The 99th percentile is under 20 miles, so 100 keeps long airport trips and removes typing errors.
- Duration between 60 seconds and 4 hours.
- Speed up to 80 mph, which removes trips that are impossible in New York.
- Pickup and dropoff in known zones.

In [5]:

start = F.lit("2024-01-01 00:00:00").cast("timestamp_ntz")
end = F.lit("2024-04-01 00:00:00").cast("timestamp_ntz")

rules = [
    ("pickup Jan-Mar 2024",  (F.col("tpep_pickup_datetime") >= start) & (F.col("tpep_pickup_datetime") < end)),
    ("distance (0, 100] mi", (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100)),
    ("duration [60 s, 4 h]", (dur >= 60) & (dur <= 4 * 3600)),
    ("speed <= 80 mph",      F.col("trip_distance") / (dur / 3600) <= 80),
    ("known zones",          ~F.col("PULocationID").isin(264, 265) & ~F.col("DOLocationID").isin(264, 265)),
]

# Apply the rules one by one and count how many trips each one removes

clean = df
n_before = df.count()
print(f"{'start':<25}{n_before:>12,}")
for name, rule in rules:
    clean = clean.filter(rule)
    n = clean.count()
    print(f"{name:<25}{n:>12,}   removed: {n_before - n:>9,} ({(n_before - n) / n_before:.2%})")
    n_before = n

start                       9,554,778


pickup Jan-Mar 2024         9,554,757   removed:        21 (0.00%)


distance (0, 100] mi        9,338,741   removed:   216,016 (2.26%)


duration [60 s, 4 h]        9,298,539   removed:    40,202 (0.43%)


speed <= 80 mph             9,298,271   removed:       268 (0.00%)


known zones                 9,213,529   removed:    84,742 (0.91%)


The cleaning keeps 9,213,529 trips and removes 3.57% of the raw data. The distance rule (216,016 trips) and the unknown zones (84,742) remove the most.

### Step 7 - Saving the Clean Base

The true duration is computed here, from the original timestamps, and saved as `duration_s_true`. It stays at full precision in every experiment, because it is the value the network is scored against.

In [6]:
clean = clean.withColumn("duration_s_true", dur)
clean.write.mode("overwrite").parquet(CLEAN)

# Read it back to check what was saved
check = spark.read.parquet(CLEAN)
print("Trips saved:", check.count())
check.select(F.min("duration_s_true").alias("min_duration_s"),
             F.max("duration_s_true").alias("max_duration_s"),
             F.min("tpep_pickup_datetime").alias("first_pickup"),
             F.max("tpep_pickup_datetime").alias("last_pickup")).show(vertical=True)

spark.stop()

Trips saved: 9213529


-RECORD 0-----------------------------
 min_duration_s | 60                  
 max_duration_s | 14344               
 first_pickup   | 2024-01-01 00:00:00 
 last_pickup    | 2024-03-31 23:59:59 



The clean base has 9,213,529 trips, with durations between 60 and 14,344 seconds and every pickup between January and March 2024.